In [ ]:
import os
import tensorflow as tf
import tensorflow.keras as keras
import numpy as np
import random
import tensorflow.keras.backend as K
import gc
import natsort
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Lambda

In [ ]:
def seed_everything(seed: int = 41):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    tf.random.set_seed(seed)

seed_everything()

**Import custom metrics and loss**

In [ ]:
import CorrelationAccuracyMetric
import CorrelationAccuracyNegMetric
import supervised_contrastive_loss

**Hyperparameters**

In [ ]:
imsize = 256
input_shape = (imsize, imsize, 2)

width = 128
kernal_size = 3
strides = 2
encoder_lr = 1e-3
num_epochs_encoder = 100

classifier_lr = 1e-3
num_epochs_classifier = 100

encoder_trainable = False

# **Stage1. Encoder**

## Model definition

In [ ]:
class SupConModel(tf.keras.Model):
    def __init__(self, encoder, projection_head, augmenter, **kwargs):
        super(SupConModel, self).__init__(**kwargs)
        self.encoder = encoder
        self.projection_head = projection_head
        self.augmenter = augmenter
        self.loss_tracker = tf.keras.metrics.Mean(name='loss')
        self.PosCorr = CorrelationAccuracyMetric()
        self.NegCorr = CorrelationAccuracyNegMetric()

    def compile(self, optimizer, loss, **kwargs):
        super(SupConModel, self).compile(**kwargs)
        self.optimizer = optimizer
        self.loss = loss

    def call(self, inputs):
        augmented_inputs = self.augmenter(inputs)
        features = self.encoder(augmented_inputs)
        return tf.nn.l2_normalize(features, axis=1)

    @tf.function
    def train_step(self, data):
        images, labels = data
        with tf.GradientTape() as tape:
            projections = self(images, training=True)
            loss = self.loss(labels, projections)
                
        trainable_vars = self.encoder.trainable_weights
        gradients = tape.gradient(loss, trainable_vars)
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))

        self.loss_tracker.update_state(loss)
        self.PosCorr.update_state(labels, projections)
        self.NegCorr.update_state(labels, projections)

        return {
            'loss': self.loss_tracker.result(),
            'corr_pos_pair': self.PosCorr.result(),
            'corr_neg_pair': self.NegCorr.result(),
        }

    @tf.function
    def test_step(self, data):
        images, labels = data
        projections = self(images, training=False)
        loss = self.loss(labels, projections)
        
        self.loss_tracker.update_state(loss)
        self.PosCorr.update_state(labels, projections)
        self.NegCorr.update_state(labels, projections)

        return {
            'loss': self.loss_tracker.result(),
            'corr_pos_pair': self.PosCorr.result(),
            'corr_neg_pair': self.NegCorr.result(),
        }

    @property
    def metrics(self):
        return [self.loss_tracker, self.PosCorr, self.NegCorr]

## Data loader

In [ ]:
def create_dataset(fpaths, labels, batch_size, buffer_size=1000):
    # Convert file paths and labels from lists to TensorFlow tensors    
    fpaths = tf.constant(fpaths, dtype=tf.string)
    labels = tf.constant(labels, dtype=tf.float32)
    
    # Create a TensorFlow Dataset from the file paths and labels
    dataset = tf.data.Dataset.from_tensor_slices((fpaths, labels))
    
    # Define the parsing function to apply the preprocessing
    def parse_function2(fpath, label):
        # Use tf.numpy_function to load the .npy file with NumPy
        def load_npy_file(f):
            return np.load(f)
        
        images = tf.numpy_function(load_npy_file, [fpath], tf.float32)
        images.set_shape((imsize,imsize,2))  # Explicitly set the shape
  
        label = tf.cast(label, tf.float32)
        label = tf.reshape(label, [1])
        
        return images, label

    # Map the parsing function to the dataset
    dataset = dataset.map(parse_function2, num_parallel_calls=tf.data.AUTOTUNE)
    
    # Shuffle, batch, and prefetch the dataset
    dataset = dataset.batch(batch_size, drop_remainder=True)
    dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)
    
    return dataset

In [ ]:
def get_all_files(folder):
    file_paths = []
    for root, dirs, files in os.walk(folder):
        for file in files:
            file_paths.append(os.path.abspath(os.path.join(root, file)))
    return file_paths

In [ ]:
HP_files = natsort.natsorted(get_all_files(f'./dataset/HP'))
HP_labels = [1]*len(os.listdir(f'./dataset/HP/Change')) + [0]*len(os.listdir(f'./dataset/HP/NoChange'))

BP_files = natsort.natsorted(get_all_files(f'./dataset/BP'))
BP_labels = [1]*len(os.listdir(f'./dataset/BP/Change')) + [0]*len(os.listdir(f'./dataset/BP/NoChange'))

In [ ]:
patIDsHP = np.unique([fileName.split('_')[-2] for fileName in HP_files])
trIds = patIDsHP[:int(0.7*len(patIDsHP))]
vaIds = patIDsHP[int(0.7*len(patIDsHP)):]

In [ ]:
HP_files_Tr = [fileName for fileName in HP_files if fileName.split('_')[-2] in trIds]
HP_files_Va = [fileName for fileName in HP_files if fileName.split('_')[-2] in vaIds]

In [ ]:
HP_labels_Tr = [0 if 'NoChange' in fileName else 1 for fileName in HP_files_Tr]
HP_labels_Va = [0 if 'NoChange' in fileName else 1 for fileName in HP_files_Va]

In [ ]:
temp = list(zip(HP_files_Tr, HP_labels_Tr))
random.shuffle(temp)
HP_files_Tr, HP_labels_Tr = zip(*temp)

temp = list(zip(HP_files_Va, HP_labels_Va))
random.shuffle(temp)
HP_files_Va, HP_labels_Va = zip(*temp)

temp = list(zip(BP_files, BP_labels))
random.shuffle(temp)
BP_files, BP_labels = zip(*temp)


In [ ]:
tr_gen = create_dataset(HP_files_Tr, HP_labels_Tr, batch_size=256)
va_gen = create_dataset(HP_files_Va, HP_labels_Va, batch_size=256)
ts_gen = create_dataset(BP_files, BP_labels, batch_size=256)

# Model build

In [ ]:
model = SupConModel(
    augmenter=keras.Sequential(
    [
        layers.Input(shape=(imsize, imsize, 2), name='AugInput'),        
        layers.RandomTranslation(height_factor=(-0.1,0.1), width_factor=(-0.1,0.1), fill_mode='constant'),
        layers.RandomRotation((-45/360, 45/360), fill_mode='constant', name='AugRandRotate'),        
    ],
    name="autmenter"),
    
    encoder=keras.Sequential(
        [
            layers.Input(shape=(imsize, imsize, 2), name='EncInput'),            
            layers.Conv2D(width, kernel_size=kernal_size, strides=strides, activation="relu", name='EncCov2_1'),
            layers.Conv2D(width, kernel_size=kernal_size, strides=strides, activation="relu", name='EncCov2_2'),  
            layers.Conv2D(width, kernel_size=kernal_size, strides=strides, activation="relu", name='EncCovFinal'),
            layers.GlobalAveragePooling2D(name='EncGAP'),
        ],
        name="encoder"),
    
    projection_head=keras.Sequential(
        [
            layers.Input(shape=(width,), name='ProjHeadInput'),
            layers.Dense(width, name='ProjHeadDense'),
        ],
        name="projection_head"),
)


In [ ]:
# Compile model with supervised contrastive loss
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=encoder_lr), 
    loss=supervised_contrastive_loss, 
    metrics=[CorrelationAccuracyMetric(), CorrelationAccuracyNegMetric()]    
    )

# Print model summary
model.summary()

# Training

In [ ]:
class SaveEncoderCallback(tf.keras.callbacks.Callback):
    def __init__(self, encoder, filepath):
        super(SaveEncoderCallback, self).__init__()
        self.encoder = encoder
        self.filepath = filepath
        self.best_loss = float('inf')

    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get('loss')
        if current_loss < self.best_loss:
            self.best_loss = current_loss
            self.encoder.save(self.filepath.format(epoch=epoch, loss=current_loss))        

mdl_Name = f'SupCon({imsize})'
mdl_path = f'./{mdl_Name}/encoder.keras'
if not os.path.exists(f'./{mdl_Name}/'):
    os.mkdir(f'./{mdl_Name}/')
mcp = SaveEncoderCallback(model.encoder, mdl_path)

In [ ]:
model.evaluate(tr_gen)

In [ ]:
# run training
history = model.fit(tr_gen, epochs=num_epochs_encoder, validation_data=va_gen, callbacks=[mcp])

In [ ]:
model = tf.keras.models.load_model(mdl_path)
model.add(layers.Lambda(lambda x: tf.nn.l2_normalize(x,axis=1), name='L2Norm'))
print(f"Best val loss at epoah {np.argmin(history.history['val_loss'])}")

---

**Test set evaluation**

In [ ]:
negCorr = CorrelationAccuracyNegMetric()
posCorr = CorrelationAccuracyMetric()
tsloss = []
scoreTest = []
labelsTsResult = []
cnt = 0
for batch in ts_gen:
    image, label = batch
    predTest = model.predict(image, verbose=False)        
    negCorr.update_state(label, predTest)
    posCorr.update_state(label, predTest)
    tsloss.append(supervised_contrastive_loss(label, predTest).numpy())
    scoreTest.extend(predTest.tolist())
    labelsTsResult.extend(label.numpy().tolist())  
    cnt += 1
    if cnt > 100:
        break

print(f'ts_corr_neg_pair: {negCorr.result():.4f} - ts_corr_pos_pair: {posCorr.result():.4f} - ts_loss: {np.mean(tsloss)}')

# **Stage 2: Classifier**

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    tf.random.set_seed(seed)

seed_everything()

## Load encoder

In [ ]:
mdl_path = f'./{mdl_Name}/encoder.keras'
encoder = tf.keras.models.load_model(mdl_path)

# Model build

In [ ]:
class L2NormalizationLayer(layers.Layer):
    def __init__(self, **kwargs):
        super(L2NormalizationLayer, self).__init__(**kwargs)

    def call(self, inputs):
        return tf.math.l2_normalize(inputs, axis=1)

    def compute_output_shape(self, input_shape):
        return input_shape

In [ ]:
encoder.trainable = encoder_trainable

input = keras.Input((imsize, imsize, 2))
x = layers.RandomRotation((-5/360, 5/360), fill_mode='constant', name='RandRotate')(input)
for layer in encoder.layers:
    x = layer(x)
x = L2NormalizationLayer(name='L2Norm')(x)
output = layers.Dense(1, activation='sigmoid', name='Sigmoid')(x)
myMdl = keras.Model(input, output)

In [ ]:
myMdl.summary()

In [ ]:
myMdl.compile(optimizer=keras.optimizers.Adam(learning_rate=classifier_lr), 
              loss=keras.losses.BinaryCrossentropy(),
              metrics=[keras.metrics.AUC(name='AUC'),'accuracy']
              )

In [ ]:
clfPath = f'./{mdl_Name}/classifier.keras'
mcp = tf.keras.callbacks.ModelCheckpoint(filepath=clfPath,
                                         monitor='val_loss',
                                         save_best_only=True,
                                         mode='min')

In [ ]:
# run training
history2 = myMdl.fit(tr_gen, epochs=num_epochs_classifier, validation_data=va_gen, callbacks=[mcp])

In [ ]:
custom_objects = {'L2NormalizationLayer': L2NormalizationLayer}
myMdl = keras.models.load_model(clfPath, custom_objects=custom_objects, safe_mode=False)

In [ ]:
myMdl.evaluate(ts_gen)